# ANI-style network for 7- and 8-atom Lennard-Jones systems

This notebook extends the original 2-atom ANI/LJ training to many-body
configurations with `N = 7` and `N = 8` atoms.

Pipeline:
1. Generate random non-overlapping configurations of 7 and 8 atoms.
2. Compute the analytic *force-shifted* Lennard-Jones energy and forces.
3. Train a single ANI-like network on a mixed batch of both system sizes
   (the model is permutation- and size-invariant by construction).
4. Evaluate predicted energies and forces on held-out test configurations.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

# Force-shifted Lennard-Jones target potential
def lj_energy_fs(positions, epsilon=1.0, sigma=1.0, r_cut=3.0):
    """
    Force-shifted Lennard-Jones energy.

    Args:
        positions: [batch, N, 3]
    Returns:
        [batch] tensor of total LJ energies.
    """
    batch, N, _ = positions.shape
    device = positions.device

    rij = positions[:, :, None, :] - positions[:, None, :, :]
    r = torch.linalg.norm(rij, dim=-1)

    eye = torch.eye(N, device=device, dtype=torch.bool)

    r = torch.where(
        eye[None, :, :],
        torch.full_like(r, r_cut + 1.0),
        r,
    )
    r_safe = torch.clamp(r, min=1.0e-8)

    sr6 = (sigma / r_safe) ** 6
    sr12 = sr6 ** 2
    V = 4.0 * epsilon * (sr12 - sr6)

    sr6_c = (sigma / r_cut) ** 6
    sr12_c = sr6_c ** 2
    V_c = 4.0 * epsilon * (sr12_c - sr6_c)

    dVdr_c = 4.0 * epsilon * (
        -12.0 * sigma**12 / r_cut**13
        + 6.0 * sigma**6 / r_cut**7
    )

    V_fs = V - V_c - (r_safe - r_cut) * dVdr_c

    mask = (r < r_cut) & (~eye[None, :, :])
    V_fs = torch.where(mask, V_fs, torch.zeros_like(V_fs))

    return 0.5 * V_fs.sum(dim=(1, 2))


In [ ]:
def cutoff(r, r_cut):
    fc = 0.5 * (torch.cos(torch.pi * r / r_cut) + 1.0)
    return torch.where(r < r_cut, fc, torch.zeros_like(r))


class RadialAEV(nn.Module):
    def __init__(self, r_cut=3.0, eta=16.0, n_radial=64):
        super().__init__()
        self.r_cut = r_cut
        self.eta = eta
        self.register_buffer("Rs", torch.linspace(0.7, r_cut, n_radial))

    def forward(self, positions):
        """
        positions: [batch, N, 3]
        returns:   [batch, N, n_radial]
        """
        batch, N, _ = positions.shape
        device = positions.device
        dtype = positions.dtype

        rij = positions[:, :, None, :] - positions[:, None, :, :]
        r = torch.linalg.norm(rij, dim=-1)

        eye = torch.eye(N, device=device, dtype=torch.bool)
        r = torch.where(
            eye[None, :, :],
            torch.full_like(r, 1.0e6),
            r,
        )

        fc = cutoff(r, self.r_cut)
        Rs = self.Rs.to(device=device, dtype=dtype)

        radial = torch.exp(-self.eta * (r[..., None] - Rs) ** 2) * fc[..., None]

        not_self = (~eye)[None, :, :, None].to(dtype)
        radial = radial * not_self

        return radial.sum(dim=2)


class ANILJ(nn.Module):
    def __init__(self, n_radial=64, n_hidden=64):
        super().__init__()
        self.aev = RadialAEV(n_radial=n_radial)
        self.atomic_net = nn.Sequential(
            nn.Linear(n_radial, n_hidden),
            nn.SiLU(),
            nn.Linear(n_hidden, n_hidden),
            nn.SiLU(),
            nn.Linear(n_hidden, 1),
        )

    def forward(self, positions):
        """
        positions: [batch, N, 3]
        returns total energy: [batch]
        """
        G = self.aev(positions)
        atomic_energies = self.atomic_net(G).squeeze(-1)
        return atomic_energies.sum(dim=1)


In [ ]:
def generate_cluster_batch(
    batch_size,
    N,
    box=2.6,
    r_min=0.90,
    max_attempts=200,
):
    """
    Generate `batch_size` configurations of N atoms in a cubic box.

    Each configuration is built by sequential rejection sampling so that no
    pair of atoms is closer than `r_min` (avoids the divergent LJ core).
    """
    positions = torch.empty(batch_size, N, 3)

    for b in range(batch_size):
        coords = torch.empty(N, 3)
        coords[0] = box * torch.rand(3)

        for i in range(1, N):
            placed = False
            for _ in range(max_attempts):
                candidate = box * torch.rand(3)
                d = torch.linalg.norm(coords[:i] - candidate, dim=-1)
                if torch.all(d >= r_min):
                    coords[i] = candidate
                    placed = True
                    break
            if not placed:
                direction = torch.randn(3)
                direction = direction / torch.linalg.norm(direction)
                coords[i] = coords[i - 1] + r_min * direction

        positions[b] = coords

    return positions


def compute_targets(positions):
    """Return analytic LJ energy and forces (both detached)."""
    pos = positions.clone().detach().requires_grad_(True)
    E = lj_energy_fs(pos)
    F_target = -torch.autograd.grad(E.sum(), pos, create_graph=False)[0]
    return E.detach(), F_target.detach()

In [ ]:
model = ANILJ(n_radial=64, n_hidden=64)
optimizer = torch.optim.Adam(model.parameters(), lr=1.0e-4)

N_VALUES = (7, 8)
BATCH_PER_SIZE = 64
N_STEPS = 4000
FORCE_WEIGHT = 0.01
PRINT_EVERY = 200

history = []

for step in range(N_STEPS):
    total_energy_loss = 0.0
    total_force_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for N in N_VALUES:
        pos_base = generate_cluster_batch(
            batch_size=BATCH_PER_SIZE,
            N=N,
            box=2.6,
            r_min=0.90,
        ).detach()

        target_E, target_F = compute_targets(pos_base)

        pos_energy = pos_base.clone().detach()
        pred_E = model(pos_energy)
        energy_loss = F.smooth_l1_loss(
            pred_E / N,
            target_E / N,
            beta=0.05,
        )

        pos_force = pos_base.clone().detach().requires_grad_(True)
        pred_E_force = model(pos_force)
        pred_F = -torch.autograd.grad(
            pred_E_force.sum(),
            pos_force,
            create_graph=True,
        )[0]
        force_loss = F.smooth_l1_loss(pred_F, target_F, beta=0.05)

        loss = energy_loss + FORCE_WEIGHT * force_loss
        loss.backward()

        total_energy_loss += energy_loss.item()
        total_force_loss += force_loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
    optimizer.step()

    history.append((step, total_energy_loss, total_force_loss))

    if step % PRINT_EVERY == 0:
        print(
            f"step={step:5d}  "
            f"energy_loss={total_energy_loss:.6f}  "
            f"force_loss={total_force_loss:.6f}"
        )

print("training done.")


/home/tigran/.local/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


step=    0  energy_loss=1.051052  force_loss=22.213144
step=  200  energy_loss=0.620242  force_loss=19.652836
step=  400  energy_loss=0.514154  force_loss=19.386855
step=  600  energy_loss=0.370523  force_loss=17.224413
step=  800  energy_loss=0.259475  force_loss=14.028044
step= 1000  energy_loss=0.192314  force_loss=13.052562
step= 1200  energy_loss=0.183997  force_loss=11.484096
step= 1400  energy_loss=0.177024  force_loss=11.221961
step= 1600  energy_loss=0.126390  force_loss=9.841440
step= 1800  energy_loss=0.090388  force_loss=7.216859
step= 2000  energy_loss=0.117787  force_loss=8.150228
step= 2200  energy_loss=0.117695  force_loss=7.130929
step= 2400  energy_loss=0.119958  force_loss=8.694102
step= 2600  energy_loss=0.072863  force_loss=5.724021
step= 2800  energy_loss=0.089908  force_loss=6.026056
step= 3000  energy_loss=0.051501  force_loss=5.118459
step= 3200  energy_loss=0.097726  force_loss=5.888482
step= 3400  energy_loss=0.067533  force_loss=5.697294
step= 3600  energy_l

In [ ]:
import matplotlib.pyplot as plt

def evaluate(model, N, n_test=512):
    pos_test = generate_cluster_batch(
        batch_size=n_test,
        N=N,
        box=2.6,
        r_min=0.90,
    )
    target_E, target_F = compute_targets(pos_test)

    pos = pos_test.clone().detach().requires_grad_(True)
    pred_E = model(pos)
    pred_F = -torch.autograd.grad(pred_E.sum(), pos)[0]

    pred_E = pred_E.detach()
    pred_F = pred_F.detach()

    e_mae = (pred_E - target_E).abs().mean().item()
    f_mae = (pred_F - target_F).abs().mean().item()

    return {
        "target_E": target_E,
        "pred_E": pred_E,
        "target_F": target_F,
        "pred_F": pred_F,
        "E_MAE": e_mae,
        "F_MAE": f_mae,
    }

fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for col, N in enumerate(N_VALUES):
    res = evaluate(model, N=N, n_test=512)
    print(
        f"N={N}: energy MAE = {res['E_MAE']:.4e}  "
        f"force MAE = {res['F_MAE']:.4e}"
    )

    ax_e = axes[0, col]
    e_true = res["target_E"].numpy()
    e_pred = res["pred_E"].numpy()
    ax_e.scatter(e_true, e_pred, s=8, alpha=0.6)
    lims = [min(e_true.min(), e_pred.min()), max(e_true.max(), e_pred.max())]
    ax_e.plot(lims, lims, "k--", linewidth=0.8)
    ax_e.set_xlabel(r"True LJ energy $E/\epsilon$")
    ax_e.set_ylabel(r"Predicted energy $E/\epsilon$")
    ax_e.set_title(f"Energy parity, N={N}  (MAE={res['E_MAE']:.2e})")
    ax_e.grid(True)

    ax_f = axes[1, col]
    f_true = res["target_F"].reshape(-1).numpy()
    f_pred = res["pred_F"].reshape(-1).numpy()
    ax_f.scatter(f_true, f_pred, s=4, alpha=0.4)
    lims = [min(f_true.min(), f_pred.min()), max(f_true.max(), f_pred.max())]
    ax_f.plot(lims, lims, "k--", linewidth=0.8)
    ax_f.set_xlabel(r"True force component $F/(\epsilon/\sigma)$")
    ax_f.set_ylabel("Predicted force component")
    ax_f.set_title(f"Force parity, N={N}  (MAE={res['F_MAE']:.2e})")
    ax_f.grid(True)

fig.tight_layout()
fig.savefig("LJ_ANI_7_8.png", dpi=120)
plt.close(fig)

print("saved plot to LJ_ANI_7_8.png")


N=7: energy MAE = 3.6262e-01  force MAE = 2.6453e+00
N=8: energy MAE = 4.3383e-01  force MAE = 2.9143e+00
saved plot to LJ_ANI_7_8.png


In [ ]:
r_values = torch.linspace(0.90, 3.0, 400)
positions = torch.zeros(len(r_values), 2, 3)
positions[:, 1, 0] = r_values

with torch.no_grad():
    E_lj = lj_energy_fs(positions)
    E_nn = model(positions)

fig = plt.figure(figsize=(7, 5))
plt.plot(r_values.numpy(), E_lj.numpy(), label="Force-shifted LJ target")
plt.plot(r_values.numpy(), E_nn.numpy(), "--", label="ANI prediction (trained on N=7,8)")
plt.axhline(0.0, linewidth=0.8)
plt.xlabel(r"Distance $r/\sigma$")
plt.ylabel(r"Energy $V(r)/\epsilon$")
plt.title("Pair-potential extrapolation from a 7/8-atom-trained ANI model")
plt.legend()
plt.grid(True)
fig.savefig("LJ_ANI_7_8_pair.png", dpi=120)
plt.close(fig)

print("saved plot to LJ_ANI_7_8_pair.png")

saved plot to LJ_ANI_7_8_pair.png
